# Neo4j Private Link Connectivity Test

Validates connectivity from Databricks serverless compute to a Neo4j Enterprise Edition
cluster over Azure Private Link. Traffic flows entirely over the Azure backbone:

```
Databricks Serverless --> NCC Private Endpoint --> Private Link Service --> Internal LB --> Neo4j EE VMs
```

**Prerequisites:**
- Private Link infrastructure deployed (`uv run setup-private-link`)
- NCC private endpoint rule created and connection approved
- NCC attached to this workspace
- Secrets stored in a Databricks secret scope (see setup below)

In [ ]:
%pip install neo4j>=5.20.0

---

## Setup: Create Secret Scope

Run these commands once from the Databricks CLI to store Neo4j credentials:

```bash
# Create the secret scope
databricks secrets create-scope neo4j-private-link

# Store credentials
databricks secrets put-secret neo4j-private-link password --string-value '<NEO4J_PASSWORD>'
```

The domain and user are not sensitive, so they're set directly in the configuration cell below.

In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Private Link domain — must match the domain in the NCC private endpoint rule
NEO4J_DOMAIN = "neo4j-ee.private.neo4j.com"

# neo4j:// (not neo4j+s://) — traffic is private via Azure Private Link, TLS optional
NEO4J_URI = f"neo4j://{NEO4J_DOMAIN}:7687"

# Credentials from Databricks secrets
SCOPE_NAME = "neo4j-private-link"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = dbutils.secrets.get(SCOPE_NAME, "password")

print("Configuration loaded:")
print(f"  Domain: {NEO4J_DOMAIN}")
print(f"  URI:    {NEO4J_URI}")
print(f"  User:   {NEO4J_USER}")

---

## Test 1: TCP Connectivity

Verifies the network path is open from serverless compute to Neo4j over Private Link on port 7687 (Bolt).

In [ ]:
import subprocess
import time

print("=" * 60)
print("TEST: TCP Connectivity over Private Link")
print("=" * 60)
print(f"\nTarget: {NEO4J_DOMAIN}:7687")

try:
    start = time.time()
    result = subprocess.run(
        ["nc", "-zv", NEO4J_DOMAIN, "7687"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=10,
    )
    elapsed = (time.time() - start) * 1000
    output = (result.stdout.decode() + result.stderr.decode()).strip()

    if result.returncode == 0:
        print(f"\n[PASS] TCP connection established in {elapsed:.1f}ms")
        if output:
            print(f"  Raw: {output}")
    else:
        print(f"\n[FAIL] Cannot reach {NEO4J_DOMAIN}:7687")
        print(f"  Output: {output}")
        print("\n  Check that:")
        print("  - NCC private endpoint status is ESTABLISHED")
        print("  - NCC is attached to this workspace")
        print("  - Domain in NCC rule matches NEO4J_DOMAIN above")

except Exception as e:
    print(f"\n[FAIL] Error: {e}")

---

## Test 2: Neo4j Driver Connectivity

Connects using the Neo4j Python driver over Bolt, authenticates, and runs a test query.

In [ ]:
from neo4j import GraphDatabase
import time

print("=" * 60)
print("TEST: Neo4j Driver over Private Link")
print("=" * 60)
print(f"\nURI: {NEO4J_URI}")

try:
    start = time.time()
    driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
    driver.verify_connectivity()
    connect_ms = (time.time() - start) * 1000
    print(f"\n[PASS] Connected and authenticated in {connect_ms:.1f}ms")

    # Test query
    records, _, _ = driver.execute_query(
        "RETURN 'Connected over Private Link' AS message"
    )
    print(f"[PASS] Query result: {records[0]['message']}")

    # Server info
    records, _, _ = driver.execute_query(
        "CALL dbms.components() YIELD name, versions RETURN name, versions"
    )
    for record in records:
        print(f"\n  Server: {record['name']} {record['versions']}")

    driver.close()
    total_ms = (time.time() - start) * 1000

    print(f"\n  Connection: {connect_ms:.1f}ms")
    print(f"  Total:      {total_ms:.1f}ms")
    print("\n" + "=" * 60)
    print("PRIVATE LINK CONNECTIVITY VERIFIED")
    print("=" * 60)

except Exception as e:
    print(f"\n[FAIL] {e}")
    print("\n  Check that:")
    print("  - Neo4j cluster is running")
    print("  - Password in secret scope is correct")
    print("  - VMSS is in the internal LB backend pool")